# PCF Estimation with Open-Weight LLM on Colab GPU

Runs the full Carbon Catalogue evaluation (866 products × 2 LLM methods) using
Qwen 2.5 3B Instruct via HuggingFace Transformers on a Colab T4 GPU.

**Runtime**: expect **~40–50 min per LLM pass** (866 × ~3 s) × 2 passes (zero-shot, then few-shot) ≈ **1.5–2 h** on T4.  
**Requirements**: Colab GPU runtime, Google Drive mounted.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import subprocess, sys, shutil

DRIVE_BASE = Path('/content/drive/MyDrive')
REPO_URL = 'https://github.com/andersvestrum/carbon-aware-recsys.git'
REPO_BRANCH = 'pcf-complete'
REPO_ROOT = Path('/content/carbon-aware-recsys')
DRIVE_RESULTS = DRIVE_BASE / 'carbon-aware-recsys-colab' / 'results' / 'carbon'

LLM_MODEL = 'Qwen/Qwen2.5-3B-Instruct'

if REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)
subprocess.run(
    ['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_ROOT)],
    check=True,
)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_ROOT / 'requirements.txt')],
    check=True,
)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', 'accelerate'],
    check=True,
)
print('Repo cloned and dependencies installed.')

In [ ]:
import warnings
import torch
import transformers
from transformers import pipeline

transformers.logging.set_verbosity_error()
warnings.filterwarnings(
    'ignore',
    message=r'Both `max_new_tokens` .* and `max_length`',
)

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

pipe = pipeline(
    'text-generation',
    model=LLM_MODEL,
    torch_dtype=torch.float16,
    device_map='auto',
)
# Qwen ships a tiny default max_length (e.g. 20); unset so only max_new_tokens applies.
gc = pipe.model.generation_config
if getattr(gc, 'max_length', None) is not None:
    gc.max_length = None
if pipe.tokenizer.pad_token_id is None:
    pipe.tokenizer.pad_token_id = pipe.tokenizer.eos_token_id

print(f'Model loaded: {LLM_MODEL}')

In [ ]:
import time

test_messages = [
    {'role': 'system', 'content': 'Estimate PCF in kg CO2e. Output your final answer as: PCF: X.X kg CO2e'},
    {'role': 'user', 'content': 'Product: Samsung 65-inch LED TV\nPCF: ?'},
]

GEN_KW = dict(max_new_tokens=384, do_sample=False)

t0 = time.time()
out = pipe(test_messages, **GEN_KW)
elapsed = time.time() - t0

reply = out[0]['generated_text'][-1]['content']
print(f'Test call: {elapsed:.1f}s')
print(f'Response (last 150 chars): ...{reply[-150:]}')
print(f'\nEstimated time for 1,732 calls: {1732 * elapsed / 60:.0f} minutes')

In [ ]:
import sys
sys.path.insert(0, str(REPO_ROOT))

# Apply patches
loader_path = REPO_ROOT / 'src' / 'data' / 'amazon_loader.py'
text = loader_path.read_text()
text = text.replace('MAX_META_ROWS = 50_000  # TODO: remove cap for full runs', 'MAX_META_ROWS = None')
loader_path.write_text(text)

retrieval_path = REPO_ROOT / 'src' / 'carbon' / 'retrieval.py'
text = retrieval_path.read_text()
text = text.replace('timeout: float = 60.0,', 'timeout: float = 300.0,')
retrieval_path.write_text(text)

print('Patches applied.')

In [ ]:
from src.data.carbon_loader import load_carbon_catalogue

df = load_carbon_catalogue()
print(f'Carbon Catalogue ready: {len(df)} products')

In [ ]:
from src.carbon.retrieval import (
    OpenAILLMClient,
    PCFRetrievalEstimator,
    RetrievalConfig,
    parse_numeric_response,
    set_global_determinism,
)


class TransformersLLMClient:
    """Drop-in replacement for OpenAILLMClient using a local HF pipeline."""

    system_prompt = OpenAILLMClient.system_prompt

    def __init__(self, hf_pipeline, model_name: str):
        self.model = model_name
        self._pipe = hf_pipeline

    @property
    def is_available(self) -> bool:
        return True

    def predict_numeric(self, prompt: str) -> float:
        messages = [
            {'role': 'system', 'content': self.system_prompt},
            {'role': 'user', 'content': prompt},
        ]
        output = self._pipe(messages, max_new_tokens=384, do_sample=False)
        text = output[0]['generated_text'][-1]['content']
        return parse_numeric_response(text)


llm_client = TransformersLLMClient(pipe, LLM_MODEL)

# Quick sanity check
val = llm_client.predict_numeric('Product: AA alkaline battery pack (4 count)\nPCF: ?')
print(f'Sanity check — battery pack PCF: {val:.1f} kg CO2e')

In [ ]:
import time
import logging
from pathlib import Path

logging.basicConfig(level=logging.INFO, format='%(asctime)s  %(levelname)s  %(message)s', datefmt='%H:%M:%S')

SEED = 42
set_global_determinism(SEED, deterministic=True, num_threads=4)

config = RetrievalConfig(
    device='cpu',
    random_seed=SEED,
    deterministic=True,
    num_threads=4,
)
estimator = PCFRetrievalEstimator(config)
estimator.fit_carbon_catalogue()

cache_path = REPO_ROOT / 'data' / 'processed' / 'carbon' / 'llm_prediction_cache.jsonl'

print('\nStarting full catalogue evaluation with LLM...')
print('(Progress bars appear below — zero_shot_llm, then few_shot_llm; ~1,732 prompts total.)\n')
t0 = time.time()

eval_predictions, eval_metrics = estimator.evaluate_on_carbon_catalogue(
    limit=None,
    random_state=SEED,
    llm_client=llm_client,
    llm_model_name=LLM_MODEL,
    llm_cache_path=cache_path,
    llm_cache_only=False,
)

elapsed_min = (time.time() - t0) / 60
print(f'\nDone in {elapsed_min:.1f} minutes')
print('\n=== Evaluation Metrics ===')
print(eval_metrics.to_string(index=False))

In [ ]:
import json

results_dir = REPO_ROOT / 'output' / 'results' / 'carbon'
results_dir.mkdir(parents=True, exist_ok=True)

eval_predictions.to_csv(results_dir / 'pcf_evaluation_predictions.csv', index=False)
eval_metrics.to_csv(results_dir / 'pcf_evaluation_metrics.csv', index=False)

run_metadata = {
    'llm_model': LLM_MODEL,
    'evaluation_rows': int(len(eval_predictions)),
    'elapsed_minutes': round(elapsed_min, 1),
    'seed': SEED,
}
with open(results_dir / 'pcf_run_metadata.json', 'w') as f:
    json.dump(run_metadata, f, indent=2)

print('Results saved locally.')
print(f'  {results_dir / "pcf_evaluation_predictions.csv"}')
print(f'  {results_dir / "pcf_evaluation_metrics.csv"}')
print(f'  {results_dir / "pcf_run_metadata.json"}')

In [ ]:
import shutil

DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)

for f in results_dir.iterdir():
    shutil.copy2(f, DRIVE_RESULTS / f.name)
    print(f'Copied {f.name} -> Drive')

if cache_path.exists():
    shutil.copy2(cache_path, DRIVE_RESULTS / cache_path.name)
    print(f'Copied {cache_path.name} -> Drive')

print('\nAll results saved to Google Drive.')